[<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>](https://colab.research.google.com/github/ADRIANVM117/quant-research-notes/blob/main/volatility_target/01_rebalancing_policies/03_economic_statistic_evaluation.ipynb)

Abra la notebook en Google Colab y seleccione **Run all**. Los historiales certificados de P0, P1 y P2 se cargan automáticamente desde este repositorio; no se requieren claves de API, Google Drive ni ejecutar notebooks anteriores.

# Volatility Targeting | Evaluación estadística y económica

## Protocolo de investigación

### Pregunta de investigación

En las etapas anteriores construimos un estimador EWMA de volatilidad y un motor causal de rebalanceo para tres políticas de exposición dinámica a SPY.

Ahora queremos determinar si las bandas de tolerancia generan una mejora económica justificable frente al rebalanceo diario y frente a alternativas estáticas.

La pregunta central es:

> ¿Pueden las bandas de rebalanceo reducir los costos de implementación del volatility targeting sin deteriorar materialmente el control del riesgo ni comprometer el crecimiento geométrico del capital?

Esta investigación distingue tres dimensiones:

- **Eficiencia operativa:** frecuencia de negociación, turnover y costos de implementación.
- **Utilidad económica:** crecimiento del patrimonio y resultados netos después de costos.
- **Control del riesgo:** volatilidad realizada, desviación respecto al objetivo y pérdidas experimentadas.

Una reducción en el número de operaciones no demuestra, por sí sola, superioridad económica. Asimismo, obtener un CAGR mayor no implica necesariamente una política de inversión superior.

Nuestro propósito es estudiar conjuntamente estas dimensiones.

### Hipótesis de investigación

Evaluaremos la siguiente hipótesis:

> Introducir bandas de tolerancia puede disminuir la fricción operativa del volatility targeting y preservar un control de riesgo económicamente aceptable frente al rebalanceo diario.

Las bandas de 5 y 10 puntos porcentuales son parámetros previamente establecidos para esta investigación. No representan valores óptimos ni serán seleccionadas retrospectivamente según su rentabilidad.

Las diferencias observadas deberán distinguirse de la incertidumbre estadística y de posibles condiciones históricas favorables.

### Políticas y benchmarks

Compararemos cinco alternativas:

| Identificador | Política | Descripción |
|---|---|---|
| P0 | Daily | Volatility targeting con rebalanceo diario hacia el peso objetivo. |
| P1 | Banda 5 pp | Rebalanceo cuando la desviación absoluta supera 5 puntos porcentuales. |
| P2 | Banda 10 pp | Rebalanceo cuando la desviación absoluta supera 10 puntos porcentuales. |
| B1 | Buy & Hold SPY | Inversión inicial del capital disponible en SPY, sin rebalanceos posteriores. |
| B2 | Asignación estática 60/40 | Asignación inicial de 60% a SPY y 40% a efectivo, permitiendo que los pesos evolucionen sin rebalanceos. |

P0 constituye la referencia principal para evaluar el valor incremental de las bandas.

B1 permite comparar las políticas dinámicas con la exposición completa al mercado.

B2 proporciona una referencia sencilla de exposición inicialmente moderada, sin aplicar pronósticos de volatilidad.

Estas alternativas no tienen necesariamente el mismo nivel de riesgo. Por ello, una comparación de rendimientos absolutos será insuficiente para determinar su utilidad relativa.

### Contrato de comparabilidad

Para evitar ventajas artificiales, todas las políticas y benchmarks deberán evaluarse bajo condiciones consistentes:

- Mismo periodo y calendario de sesiones.
- Mismo capital inicial: 200,000 MXN, convertido a USD bajo la convención documentada.
- Misma moneda contable para comparar resultados.
- Mismo tratamiento de dividendos y efectivo.
- Costos de implementación explícitos y aplicables según las operaciones de cada alternativa.
- Ausencia de información futura en decisiones de inversión.
- Trazabilidad entre las series de rendimientos y los estados de cartera que las originan.

Los resultados de las políticas P0, P1 y P2 procederán del motor desarrollado en la notebook 02. Los nuevos benchmarks deberán respetar las mismas convenciones económicas.

### Alcance de la evidencia

La muestra histórica ya fue utilizada durante el desarrollo y la exploración de las políticas.

Por tanto, los resultados de esta notebook constituirán inicialmente evidencia retrospectiva, no confirmación independiente fuera de muestra.

Las futuras pruebas de estabilidad e incertidumbre servirán para evaluar la solidez de los hallazgos, sin eliminar las limitaciones derivadas de haber inspeccionado previamente el periodo histórico.

**Objetivo del bloque:** establecer referencias comparables y un protocolo de evaluación antes de calcular resultados económicos o realizar inferencia estadística.

## Bloque 05 — Construcción de benchmarks

Este bloque construirá y verificará los insumos necesarios para comparar P0, P1, P2, B1 y B2 bajo condiciones económicas consistentes.

| Sección | Contenido |
| ------- | ------------------------------------------------------ |
| 5.1 | Insumos reproducibles y validación de P0/P1/P2 |
| 5.2 | Benchmark B1 — Buy & Hold SPY |
| 5.3 | Benchmark B2 — Asignación estática 60/40 |
| 5.4 | Validación de comparabilidad de las cinco alternativas |

### 5.1. Insumos reproducibles y validación de historiales

Las políticas P0, P1 y P2 se generaron con el motor causal de la notebook 02 y se preparan como historiales Parquet certificados. Cada archivo conserva el estado diario de cartera, las operaciones, costos, dividendos y efectivo. El manifiesto documenta sus parámetros, convenciones y hashes SHA-256.

La siguiente celda busca primero los artefactos versionados en el repositorio local. Si no están disponibles, los descarga desde GitHub. Antes de cargarlos verifica su integridad y que los tres historiales compartan el mismo contrato de evaluación.

In [ ]:
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlretrieve
import hashlib
import importlib.util
import json
import math
import subprocess
import sys

if importlib.util.find_spec('polars') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'polars==1.44.2'])

import polars as pl

ARTIFACT_FILENAMES = {
    'P0 — Daily': 'policy_p0.parquet',
    'P1 — Banda 5 pp': 'policy_p1.parquet',
    'P2 — Banda 10 pp': 'policy_p2.parquet',
}
MANIFEST_FILENAME = 'manifest.json'
REMOTE_ARTIFACTS_URL = (
    'https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/main/'
    'volatility_target/artifacts/evaluation_inputs'
)
ESSENTIAL_COLUMNS = {
    'policy', 'session_date', 'signal_date', 'target_weight',
    'effective_weight_close', 'shares', 'cash', 'dividend_receivable',
    'cash_interest', 'dividend_net', 'dividend_payment', 'open_price',
    'close_price', 'executed_shares', 'trade_executed', 'trade_notional',
    'turnover_notional', 'commission', 'iva', 'total_cost',
    'portfolio_value_open', 'portfolio_value_close',
}

def find_local_artifact_dir() -> Path | None:
    required = [MANIFEST_FILENAME, *ARTIFACT_FILENAMES.values()]
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidates = (
            folder / 'volatility_target' / 'artifacts' / 'evaluation_inputs',
            folder / 'artifacts' / 'evaluation_inputs',
        )
        for candidate in candidates:
            if all((candidate / filename).is_file() for filename in required):
                return candidate
    return None

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1_048_576), b''):
            digest.update(chunk)
    return digest.hexdigest()

artifact_dir = find_local_artifact_dir()
if artifact_dir is None:
    artifact_dir = Path.cwd() / 'evaluation_inputs'
    artifact_dir.mkdir(parents=True, exist_ok=True)
    for filename in [MANIFEST_FILENAME, *ARTIFACT_FILENAMES.values()]:
        destination = artifact_dir / filename
        temporary = destination.with_suffix(destination.suffix + '.part')
        try:
            urlretrieve(f'{REMOTE_ARTIFACTS_URL}/{filename}', temporary)
            temporary.replace(destination)
        except URLError as error:
            raise RuntimeError(
                'No fue posible descargar los insumos de evaluación desde GitHub. '
                'Verifica que artifacts/evaluation_inputs esté publicado.'
            ) from error

manifest_path = artifact_dir / MANIFEST_FILENAME
try:
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
except (OSError, json.JSONDecodeError) as error:
    raise RuntimeError('manifest.json no se pudo leer como un manifiesto válido.') from error

if manifest.get('manifest_version') != 1:
    raise ValueError('La versión del manifiesto no es compatible.')
if manifest.get('artifact_type') != 'volatility_target_policy_histories':
    raise ValueError('El manifiesto no corresponde a historiales de volatility targeting.')

parameters = manifest.get('strategy_parameters', {})
expected_parameters = {
    'ewma_lambda': 0.94,
    'target_volatility_annualized': 0.12,
    'max_exposure': 1.0,
}
for name, expected in expected_parameters.items():
    if not math.isclose(float(parameters.get(name, float('nan'))), expected):
        raise ValueError(f'El parámetro {name} no coincide con la convención documentada.')
expected_bands = {'P0 — Daily': None, 'P1 — Banda 5 pp': 0.05, 'P2 — Banda 10 pp': 0.10}
if parameters.get('bands') != expected_bands:
    raise ValueError('Las bandas del manifiesto no coinciden con P0, P1 y P2.')

conventions = manifest.get('economic_conventions', {})
required_conventions = {
    'decision_and_execution', 'slippage_bps', 'dividend_withholding',
    'cash_interest', 'stock_commission_per_share_usd',
}
if required_conventions.difference(conventions):
    raise ValueError('El manifiesto no documenta todas las convenciones económicas.')

certificates = manifest.get('policies', {})
policy_histories = {}
reference_dates = None
summary_rows = []

for policy, filename in ARTIFACT_FILENAMES.items():
    certificate = certificates.get(policy)
    if not certificate:
        raise ValueError(f'El manifiesto no contiene el certificado de {policy}.')
    path = artifact_dir / filename
    if not path.is_file():
        raise FileNotFoundError(f'Falta el historial requerido: {path.name}')
    if sha256_file(path) != certificate.get('sha256'):
        raise ValueError(f'Falló la verificación SHA-256 de {path.name}.')

    history = pl.read_parquet(path)
    missing = ESSENTIAL_COLUMNS.difference(history.columns)
    if missing:
        raise ValueError(f'{policy}: faltan columnas esenciales: {sorted(missing)}')
    if history.is_empty():
        raise ValueError(f'{policy}: el historial está vacío.')
    nulls_in_essential_columns = history.select([
        pl.col(column).is_null().sum().alias(column) for column in ESSENTIAL_COLUMNS
    ])
    if sum(nulls_in_essential_columns.row(0)) != 0:
        raise ValueError(f'{policy}: existen valores faltantes en columnas esenciales.')
    if history['policy'].n_unique() != 1 or history['policy'][0] != policy:
        raise ValueError(f'{policy}: el identificador de política es inconsistente.')

    dates = history['session_date'].to_list()
    if dates != sorted(dates) or len(dates) != len(set(dates)):
        raise ValueError(f'{policy}: las fechas deben ser únicas y ordenadas.')
    if not (history['portfolio_value_close'] > 0).all():
        raise ValueError(f'{policy}: se detectó un patrimonio no positivo.')
    if history.height != certificate.get('sessions'):
        raise ValueError(f'{policy}: el número de sesiones no coincide con su certificado.')
    if str(dates[0]) != certificate.get('first_date') or str(dates[-1]) != certificate.get('last_date'):
        raise ValueError(f'{policy}: las fechas no coinciden con su certificado.')
    if not math.isclose(
        float(history['portfolio_value_open'][0]),
        float(certificate.get('initial_portfolio_value_usd')),
        rel_tol=0.0,
        abs_tol=1e-8,
    ):
        raise ValueError(f'{policy}: el patrimonio inicial no coincide con su certificado.')
    if not math.isclose(
        float(history['portfolio_value_close'][-1]),
        float(certificate.get('final_portfolio_value')),
        rel_tol=0.0,
        abs_tol=1e-8,
    ):
        raise ValueError(f'{policy}: el patrimonio final no coincide con el certificado de la notebook 02.')

    if reference_dates is None:
        reference_dates = dates
    elif dates != reference_dates:
        raise ValueError('P0, P1 y P2 no comparten el mismo calendario de sesiones.')
    policy_histories[policy] = history
    summary_rows.append({
        'policy': policy,
        'sessions': history.height,
        'first_date': str(dates[0]),
        'last_date': str(dates[-1]),
        'initial_capital_usd': float(manifest['initial_capital']['net_usd']),
        'final_portfolio_value': float(history['portfolio_value_close'][-1]),
    })

if (
    len(reference_dates) != manifest['period']['sessions']
    or str(reference_dates[0]) != manifest['period']['first_date']
    or str(reference_dates[-1]) != manifest['period']['last_date']
):
    raise ValueError('El calendario común no coincide con el periodo documentado.')

policy_p0 = policy_histories['P0 — Daily']
policy_p1 = policy_histories['P1 — Banda 5 pp']
policy_p2 = policy_histories['P2 — Banda 10 pp']
validation_summary = pl.DataFrame(summary_rows)
validation_summary

### 5.2. Benchmark B1 — Buy & Hold SPY

Buy & Hold representa una inversión pasiva: se intenta invertir inicialmente el capital disponible en SPY y, tras la compra, la posición se conserva sin rebalanceos. Es una referencia de exposición al mercado que permite contextualizar las políticas dinámicas frente a una alternativa sin ajuste de volatilidad.

La decisión inicial fija un objetivo de 100% en SPY, sin apalancamiento ni posiciones cortas. La orden comparte con P0 la convención causal limit-on-open (LOO), su límite, slippage y comisiones. Si una compra inicial no llena, el efectivo se conserva y al cierre de esa sesión se prepara un nuevo intento para la apertura siguiente con el cierre y límite disponibles entonces. Una vez ejecutada la compra, no se generan nuevas operaciones ni se reinvierten dividendos automáticamente.

El pronóstico que interviene en el límite LOO no modifica la exposición de B1: cumple únicamente una función operativa compartida con P0. Por construir una exposición inicial completa al mercado, B1 no tiene necesariamente el mismo perfil de riesgo que las políticas dinámicas.

In [ ]:
from collections import defaultdict
from io import BytesIO
from urllib.error import URLError
from urllib.request import urlopen, urlretrieve

if 'policy_p0' not in globals() or 'manifest' not in globals():
    raise RuntimeError('La sección 5.1 debe ejecutarse antes de construir B1.')

SPY_URL = (
    'https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/main/'
    'volatility_target/series/SPY.parquet'
)
DIVIDEND_CALENDAR_FILENAME = 'spy_dividend_calendar.csv'
DIVIDEND_CALENDAR_URL = (
    'https://raw.githubusercontent.com/ADRIANVM117/quant-research-notes/'
    '280837ae0bf56571d732b745906e837cbc58f5cc/volatility_target/'
    '01_rebalancing_policies/spy_dividend_calendar.csv'
)
DIVIDEND_CALENDAR_SHA256 = (
    'a7104bf1f72e238197398a68ddeb81615fa22b967a0a2c425800b971df9d5315'
)
FRED_DFF_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFF'

B1_TARGET_WEIGHT = 1.0
ECONOMIC = manifest['economic_conventions']
SLIPPAGE_BPS = float(ECONOMIC['slippage_bps'])
IBKR_COMMISSION_PER_SHARE_USD = float(ECONOMIC['stock_commission_per_share_usd'])
IBKR_MIN_COMMISSION_USD = float(ECONOMIC['stock_minimum_commission_usd'])
IBKR_MAX_COMMISSION_RATE = float(ECONOMIC['stock_maximum_commission_rate'])
IVA_ON_COMMISSION = float(ECONOMIC['iva_on_commission'])
DIVIDEND_WITHHOLDING = float(ECONOMIC['dividend_withholding'])
IBKR_CASH_THRESHOLD_USD = float(ECONOMIC['cash_threshold_usd'])
IBKR_CASH_FULL_NAV_USD = float(ECONOMIC['cash_full_nav_usd'])
IBKR_CASH_SPREAD = float(ECONOMIC['cash_spread'])
INITIAL_CAPITAL_USD = float(manifest['initial_capital']['net_usd'])

def find_local_spy() -> Path | None:
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder, folder / 'volatility_target'):
            path = candidate / 'series' / 'SPY.parquet'
            if path.is_file():
                return path
    return None

def find_local_dividend_calendar() -> Path | None:
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidates = (
            folder / DIVIDEND_CALENDAR_FILENAME,
            folder / 'volatility_target' / '01_rebalancing_policies' / DIVIDEND_CALENDAR_FILENAME,
        )
        for candidate in candidates:
            if candidate.is_file():
                return candidate
    return None

spy_path = find_local_spy()
if spy_path is None:
    spy_path = Path.cwd() / 'series' / 'SPY.parquet'
    temporary_path = spy_path.with_suffix('.part')
    spy_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urlretrieve(SPY_URL, temporary_path)
        temporary_path.replace(spy_path)
    except URLError as error:
        raise RuntimeError('No fue posible descargar SPY.parquet desde GitHub.') from error

spy = pl.read_parquet(spy_path)
required_spy_columns = {
    'session_date', 'open', 'close', 'adjusted_close', 'dividend_amount',
}
if spy.is_empty() or required_spy_columns.difference(spy.columns):
    raise ValueError('SPY.parquet no cumple el contrato Gold requerido por B1.')

calendar_path = find_local_dividend_calendar()
if calendar_path is not None:
    dividend_calendar_bytes = calendar_path.read_bytes()
else:
    try:
        with urlopen(DIVIDEND_CALENDAR_URL, timeout=30) as response:
            dividend_calendar_bytes = response.read()
    except URLError as error:
        raise RuntimeError('No fue posible descargar el calendario de dividendos.') from error

if hashlib.sha256(dividend_calendar_bytes).hexdigest() != DIVIDEND_CALENDAR_SHA256:
    raise ValueError('El hash SHA-256 del calendario de dividendos no coincide.')

dividend_calendar = (
    pl.read_csv(BytesIO(dividend_calendar_bytes))
    .with_columns(
        pl.col('ex_dividend_date').str.to_date('%Y-%m-%d'),
        pl.col('payment_date').str.to_date('%Y-%m-%d'),
        pl.col('dividend_amount').cast(pl.Float64),
    )
    .sort('ex_dividend_date')
)
calendar_columns = {'ex_dividend_date', 'payment_date', 'dividend_amount', 'source'}
if calendar_columns.difference(dividend_calendar.columns) or dividend_calendar.height != 35:
    raise ValueError('El calendario de dividendos no cumple el contrato documentado.')

# La agenda de P0 aporta únicamente fechas, señales y límites causales.
# Los precios y dividendos de B1 provienen directamente de SPY Gold.
b1_inputs = (
    policy_p0.select('session_date', 'signal_date', 'signal_close', 'sigma_daily', 'limit_price')
    .join(
        spy.select(
            'session_date',
            pl.col('open').alias('gold_open'),
            pl.col('close').alias('gold_close'),
            pl.col('dividend_amount').alias('dividend_per_share'),
        ),
        on='session_date',
        how='inner',
    )
    .sort('session_date')
)
if b1_inputs.height != policy_p0.height or b1_inputs['session_date'].to_list() != policy_p0['session_date'].to_list():
    raise ValueError('B1 no comparte exactamente el calendario operativo de P0.')
if not (b1_inputs['signal_date'] < b1_inputs['session_date']).all():
    raise ValueError('Se detectó una señal no causal en el calendario de B1.')

p0_prices = policy_p0.select('session_date', 'open_price', 'close_price')
price_check = b1_inputs.join(p0_prices, on='session_date', how='inner').select(
    (pl.col('gold_open') - pl.col('open_price')).abs().max().alias('max_open_difference'),
    (pl.col('gold_close') - pl.col('close_price')).abs().max().alias('max_close_difference'),
    (pl.col('limit_price') - pl.col('signal_close') * (1 + pl.col('sigma_daily'))).abs().max().alias('max_limit_difference'),
).row(0, named=True)
if any(value is None or value > 1e-10 for value in price_check.values()):
    raise ValueError('Los precios o límites de B1 no coinciden con las convenciones de P0.')

def load_fred_dff():
    try:
        with urlopen(FRED_DFF_URL, timeout=30) as response:
            raw_dff = pl.read_csv(BytesIO(response.read()), null_values='.')
    except URLError as error:
        raise RuntimeError('No fue posible descargar DFF desde FRED.') from error
    date_column, rate_column = raw_dff.columns[:2]
    return (
        raw_dff.select(
            pl.col(date_column).cast(pl.Utf8).str.to_date('%Y-%m-%d').alias('dff_date'),
            pl.col(rate_column).cast(pl.Float64, strict=False).alias('dff_percent'),
        )
        .drop_nulls()
        .sort('dff_date')
    )

dff = load_fred_dff()
dff_for_sessions = (
    b1_inputs.select('session_date')
    .join_asof(dff, left_on='session_date', right_on='dff_date', strategy='backward')
    .with_columns(pl.col('dff_percent').shift(1).fill_null(0.0).alias('dff_percent_available'))
    .select('session_date', 'dff_percent_available')
)
b1_inputs = b1_inputs.join(dff_for_sessions, on='session_date', how='inner').sort('session_date')
if b1_inputs.null_count().select(pl.sum_horizontal(pl.all())).item() != 0:
    raise ValueError('Los insumos diarios de B1 contienen valores faltantes.')

local_dividend_events = b1_inputs.filter(pl.col('dividend_per_share') > 0).select(
    pl.col('session_date').alias('ex_dividend_date'),
    pl.col('dividend_per_share').alias('gold_dividend_amount'),
)
reconciled_dividends = local_dividend_events.join(dividend_calendar, on='ex_dividend_date', how='inner')
if (
    local_dividend_events.height != dividend_calendar.height
    or reconciled_dividends.height != dividend_calendar.height
    or local_dividend_events.join(dividend_calendar, on='ex_dividend_date', how='anti').height
    or dividend_calendar.join(local_dividend_events, on='ex_dividend_date', how='anti').height
    or not reconciled_dividends.select(
        (pl.col('gold_dividend_amount').round(4) == pl.col('dividend_amount').round(4)).all()
    ).item()
):
    raise ValueError('Los dividendos Gold no conciliaron con el calendario documentado.')

session_dates = b1_inputs['session_date'].to_list()
payment_available_by_ex_date = {}
for event in dividend_calendar.iter_rows(named=True):
    available_date = next((date for date in session_dates if date > event['payment_date']), None)
    if available_date is None:
        raise ValueError('No existe sesión posterior para acreditar un dividendo.')
    payment_available_by_ex_date[event['ex_dividend_date']] = available_date

def ibkr_stock_cost(trade_shares, fill_price):
    shares_traded = abs(trade_shares)
    notional = shares_traded * fill_price
    if shares_traded == 0:
        return {'commission': 0.0, 'iva': 0.0, 'total_cost': 0.0}
    commission = min(
        IBKR_MAX_COMMISSION_RATE * notional,
        max(IBKR_MIN_COMMISSION_USD, IBKR_COMMISSION_PER_SHARE_USD * shares_traded),
    )
    iva = commission * IVA_ON_COMMISSION
    return {'commission': commission, 'iva': iva, 'total_cost': commission + iva}

def maximum_buy_shares(cash, limit_price):
    lower_shares, upper_shares = 0.0, cash / limit_price
    for _ in range(100):
        candidate_shares = (lower_shares + upper_shares) / 2
        costs = ibkr_stock_cost(candidate_shares, limit_price)
        if candidate_shares * limit_price + costs['total_cost'] <= cash:
            lower_shares = candidate_shares
        else:
            upper_shares = candidate_shares
    return lower_shares

def prepare_b1_buy(cash, signal_date, signal_close, sigma_daily):
    limit_price = signal_close * (1 + sigma_daily)
    lower_shares = 0.0
    upper_shares = maximum_buy_shares(cash, limit_price)

    def state_for(shares_after):
        costs = ibkr_stock_cost(shares_after, limit_price)
        cash_after = cash - shares_after * limit_price - costs['total_cost']
        portfolio_after = shares_after * limit_price + cash_after
        return shares_after * limit_price / portfolio_after

    for _ in range(100):
        midpoint = (lower_shares + upper_shares) / 2
        if state_for(midpoint) < B1_TARGET_WEIGHT:
            lower_shares = midpoint
        else:
            upper_shares = midpoint

    order_quantity = max(0.0, (lower_shares + upper_shares) / 2)
    planned_cost = ibkr_stock_cost(order_quantity, limit_price)
    cash_reserved = order_quantity * limit_price + planned_cost['total_cost']
    if cash_reserved > cash + 1e-8:
        raise RuntimeError('La reserva de B1 excede el efectivo disponible.')
    return {
        'decision_date': signal_date,
        'signal_close': signal_close,
        'sigma_daily': sigma_daily,
        'limit_price': limit_price,
        'order_quantity': order_quantity,
        'cash_available_at_decision': cash,
        'cash_reserved': cash_reserved,
    }

def execute_b1_buy(order, cash, open_price):
    if open_price > order['limit_price']:
        return {
            'shares': 0.0, 'cash': cash, 'fill_price': None, 'no_fill': True,
            'trade_notional': 0.0, 'commission': 0.0, 'iva': 0.0, 'total_cost': 0.0,
        }
    fill_price = min(open_price * (1 + SLIPPAGE_BPS / 10_000), order['limit_price'])
    trade_shares = order['order_quantity']
    costs = ibkr_stock_cost(trade_shares, fill_price)
    cash_after = cash - trade_shares * fill_price - costs['total_cost']
    if cash_after < -1e-8:
        raise RuntimeError('La compra de B1 requeriría efectivo negativo.')
    return {
        'shares': trade_shares, 'cash': cash_after, 'fill_price': fill_price, 'no_fill': False,
        'trade_notional': trade_shares * fill_price, **costs,
    }

def cash_interest(cash, portfolio_value, dff_percent):
    eligible_cash = max(0.0, cash - IBKR_CASH_THRESHOLD_USD)
    nav_scale = min(1.0, portfolio_value / IBKR_CASH_FULL_NAV_USD)
    annual_rate = max(0.0, dff_percent / 100 - IBKR_CASH_SPREAD) * nav_scale
    return eligible_cash * annual_rate / 360

rows = list(b1_inputs.iter_rows(named=True))
shares, cash, dividend_receivable = 0.0, INITIAL_CAPITAL_USD, 0.0
pending_dividend_payments = defaultdict(float)
records = []
investment_date = None
planned_order = prepare_b1_buy(
    cash, rows[0]['signal_date'], rows[0]['signal_close'], rows[0]['sigma_daily']
)

for session_index, row in enumerate(rows):
    if planned_order is not None and planned_order['decision_date'] != row['signal_date']:
        raise RuntimeError('La orden B1 no corresponde a la señal causal de la sesión.')

    open_price, close_price = row['gold_open'], row['gold_close']
    value_before_interest = shares * open_price + cash + dividend_receivable
    daily_interest = (
        cash_interest(cash, value_before_interest, row['dff_percent_available'])
        if session_index > 0 else 0.0
    )
    cash += daily_interest

    dividend_payment = pending_dividend_payments.pop(row['session_date'], 0.0)
    nav_before_payment = shares * open_price + cash + dividend_receivable
    cash += dividend_payment
    dividend_receivable -= dividend_payment
    nav_after_payment = shares * open_price + cash + dividend_receivable
    if not math.isclose(nav_before_payment, nav_after_payment, abs_tol=1e-8):
        raise RuntimeError('El cobro de dividendos de B1 no conserva el NAV.')

    shares_before_ex_date = shares
    dividend_net = shares_before_ex_date * row['dividend_per_share'] * (1 - DIVIDEND_WITHHOLDING)
    if dividend_net:
        payment_date = payment_available_by_ex_date.get(row['session_date'])
        if payment_date is None:
            raise RuntimeError('Falta una fecha de disponibilidad para un dividendo B1.')
        dividend_receivable += dividend_net
        pending_dividend_payments[payment_date] += dividend_net

    value_open_before_execution = shares * open_price + cash + dividend_receivable
    effective_weight_open_before = (
        shares * open_price / value_open_before_execution
        if value_open_before_execution > 0 else 0.0
    )
    order = planned_order
    if order is not None:
        execution = execute_b1_buy(order, cash, open_price)
        if execution['no_fill']:
            trade_executed = False
        else:
            shares = execution['shares']
            cash = execution['cash']
            trade_executed = True
            investment_date = row['session_date']
    else:
        execution = {
            'fill_price': None, 'no_fill': False, 'trade_notional': 0.0,
            'commission': 0.0, 'iva': 0.0, 'total_cost': 0.0,
        }
        trade_executed = False

    portfolio_value_open = shares * open_price + cash + dividend_receivable
    portfolio_value_close = shares * close_price + cash + dividend_receivable
    effective_weight_close = shares * close_price / portfolio_value_close
    records.append({
        'session_date': row['session_date'],
        'signal_date': row['signal_date'],
        'order_decision_date': order['decision_date'] if order is not None else None,
        'target_weight': B1_TARGET_WEIGHT,
        'signal_close': row['signal_close'],
        'sigma_daily_for_limit': row['sigma_daily'],
        'limit_price': order['limit_price'] if order is not None else None,
        'cash_available_at_decision': order['cash_available_at_decision'] if order is not None else None,
        'cash_reserved': order['cash_reserved'] if order is not None else 0.0,
        'order_requested': order is not None,
        'order_side': 'buy' if order is not None else 'none',
        'order_quantity': order['order_quantity'] if order is not None else 0.0,
        'no_fill': execution['no_fill'],
        'trade_executed': trade_executed,
        'executed_shares': execution['shares'] if trade_executed else 0.0,
        'fill_price': execution['fill_price'],
        'trade_notional': execution['trade_notional'],
        'commission': execution['commission'],
        'iva': execution['iva'],
        'total_cost': execution['total_cost'],
        'open_price': open_price,
        'close_price': close_price,
        'shares_before_ex_date': shares_before_ex_date,
        'shares': shares,
        'cash': cash,
        'dividend_per_share': row['dividend_per_share'],
        'dividend_net': dividend_net,
        'dividend_payment': dividend_payment,
        'dividend_receivable': dividend_receivable,
        'cash_interest': daily_interest,
        'nav_before_payment': nav_before_payment,
        'nav_after_payment': nav_after_payment,
        'external_cash_flow': 0.0,
        'portfolio_value_open': portfolio_value_open,
        'portfolio_value_close': portfolio_value_close,
        'effective_weight_open_before': effective_weight_open_before,
        'effective_weight_close': effective_weight_close,
    })

    if trade_executed:
        planned_order = None
    elif order is not None:
        if session_index + 1 >= len(rows):
            raise RuntimeError('La orden inicial de B1 nunca se ejecutó dentro de la muestra.')
        next_row = rows[session_index + 1]
        planned_order = prepare_b1_buy(
            cash, next_row['signal_date'], next_row['signal_close'], next_row['sigma_daily']
        )

benchmark_b1 = pl.DataFrame(records)

# Validaciones del contrato contable y operativo de B1.
if benchmark_b1['session_date'].to_list() != policy_p0['session_date'].to_list():
    raise ValueError('B1 no conserva el calendario común de P0/P1/P2.')
if benchmark_b1['session_date'].to_list() != sorted(benchmark_b1['session_date'].to_list()):
    raise ValueError('Las fechas de B1 no están ordenadas.')
if benchmark_b1['session_date'].n_unique() != benchmark_b1.height:
    raise ValueError('B1 contiene fechas duplicadas.')
orders = benchmark_b1.filter(pl.col('order_requested'))
if not (orders['order_decision_date'] < orders['session_date']).all():
    raise ValueError('B1 usó información futura para una orden.')
if benchmark_b1['trade_executed'].sum() != 1 or investment_date is None:
    raise ValueError('B1 debe ejecutar exactamente una compra inicial.')
investment_index = benchmark_b1.filter(pl.col('trade_executed')).row(0, named=True)['session_date']
after_investment = benchmark_b1.filter(pl.col('session_date') >= investment_index)
if after_investment['shares'].n_unique() != 1 or after_investment['order_requested'].sum() != 1:
    raise ValueError('B1 generó operaciones posteriores a la compra inicial.')
if (
    not (benchmark_b1['shares'] >= 0).all()
    or not (benchmark_b1['cash'] >= -1e-8).all()
    or not (benchmark_b1['portfolio_value_close'] > 0).all()
    or not benchmark_b1['effective_weight_close'].is_between(0, 1).all()
):
    raise ValueError('B1 incumple límites de posición, efectivo o patrimonio.')
if not benchmark_b1.select(
    (pl.col('dividend_net') == pl.col('shares_before_ex_date') * pl.col('dividend_per_share') * (1 - DIVIDEND_WITHHOLDING)).all()
).item():
    raise ValueError('B1 contabilizó un dividendo de forma inconsistente.')
if (
    not math.isclose(benchmark_b1['dividend_net'].sum(), benchmark_b1['dividend_payment'].sum(), abs_tol=1e-8)
    or not math.isclose(benchmark_b1['dividend_receivable'][-1], 0.0, abs_tol=1e-8)
    or not math.isclose(benchmark_b1['external_cash_flow'].sum(), 0.0, abs_tol=1e-12)
):
    raise ValueError('B1 presenta doble conteo de dividendos o flujos externos.')
if not benchmark_b1.select(
    (pl.col('nav_before_payment') - pl.col('nav_after_payment')).abs().max() <= 1e-8
).item():
    raise ValueError('Un cobro de dividendos no conservó el NAV de B1.')

benchmark_b1_summary = pl.DataFrame([{
    'sessions': benchmark_b1.height,
    'investment_date': str(investment_date),
    'trades_executed': int(benchmark_b1['trade_executed'].sum()),
    'initial_capital_usd': INITIAL_CAPITAL_USD,
    'initial_portfolio_value_usd': benchmark_b1['portfolio_value_open'][0],
    'final_portfolio_value': benchmark_b1['portfolio_value_close'][-1],
    'accounting_validated': True,
}])
benchmark_b1_summary

### 5.3. Benchmark B2 — Asignación estática 60/40

B2 representa una asignación inicial de 60% a SPY y 40% a efectivo. Es un benchmark sencillo de exposición inicialmente moderada, útil para distinguir el efecto de una política dinámica frente a una cartera que no responde a nuevos pronósticos. Aquí 60/40 significa SPY/efectivo, no una combinación de acciones y bonos.

La compra inicial se prepara de forma causal hacia un objetivo de 60% en SPY. Después no se rebalancea: las acciones permanecen constantes, los dividendos no se reinvierten automáticamente y el efectivo evoluciona según sus reglas contables. Por ello, el peso efectivo en SPY puede aumentar o disminuir con los precios, dividendos y efectivo acumulado.

El 60% es una elección de referencia para esta investigación; no fue optimizado retrospectivamente ni representa una recomendación de asignación.

In [ ]:
if 'benchmark_b1' not in globals() or 'b1_inputs' not in globals():
    raise RuntimeError('La sección 5.2 debe ejecutarse antes de construir B2.')

B2_INITIAL_TARGET_WEIGHT = 0.60

# B2 reutiliza los insumos y convenciones ya verificados para B1.
# El hash permite confirmar que esta sección no altera el historial de B1.
def dataframe_sha256(frame):
    payload = json.dumps(
        frame.to_dicts(), default=str, ensure_ascii=False, separators=(',', ':')
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

b1_hash_before_b2 = dataframe_sha256(benchmark_b1)

def prepare_static_buy(cash, signal_date, signal_close, sigma_daily, initial_target_weight):
    """Prepara una única compra LOO hacia un objetivo estático inicial."""
    if not 0 < initial_target_weight <= 1:
        raise ValueError('El objetivo estático debe estar entre 0 y 1.')
    limit_price = signal_close * (1 + sigma_daily)
    lower_shares = 0.0
    upper_shares = maximum_buy_shares(cash, limit_price)

    def effective_weight_for(shares_after):
        costs = ibkr_stock_cost(shares_after, limit_price)
        cash_after = cash - shares_after * limit_price - costs['total_cost']
        portfolio_after = shares_after * limit_price + cash_after
        return shares_after * limit_price / portfolio_after

    for _ in range(100):
        midpoint = (lower_shares + upper_shares) / 2
        if effective_weight_for(midpoint) < initial_target_weight:
            lower_shares = midpoint
        else:
            upper_shares = midpoint

    order_quantity = max(0.0, (lower_shares + upper_shares) / 2)
    planned_cost = ibkr_stock_cost(order_quantity, limit_price)
    cash_reserved = order_quantity * limit_price + planned_cost['total_cost']
    if cash_reserved > cash + 1e-8:
        raise RuntimeError('La reserva de compra de B2 excede el efectivo disponible.')
    return {
        'decision_date': signal_date,
        'signal_close': signal_close,
        'sigma_daily': sigma_daily,
        'limit_price': limit_price,
        'order_quantity': order_quantity,
        'cash_available_at_decision': cash,
        'cash_reserved': cash_reserved,
    }

b2_rows = list(b1_inputs.iter_rows(named=True))
b2_shares, b2_cash, b2_dividend_receivable = 0.0, INITIAL_CAPITAL_USD, 0.0
b2_pending_dividend_payments = defaultdict(float)
b2_records = []
b2_investment_date = None
b2_planned_order = prepare_static_buy(
    b2_cash,
    b2_rows[0]['signal_date'],
    b2_rows[0]['signal_close'],
    b2_rows[0]['sigma_daily'],
    B2_INITIAL_TARGET_WEIGHT,
)

for session_index, row in enumerate(b2_rows):
    if b2_planned_order is not None and b2_planned_order['decision_date'] != row['signal_date']:
        raise RuntimeError('La orden B2 no corresponde a la señal causal de la sesión.')

    open_price, close_price = row['gold_open'], row['gold_close']
    value_before_interest = b2_shares * open_price + b2_cash + b2_dividend_receivable
    daily_interest = (
        cash_interest(b2_cash, value_before_interest, row['dff_percent_available'])
        if session_index > 0 else 0.0
    )
    b2_cash += daily_interest

    dividend_payment = b2_pending_dividend_payments.pop(row['session_date'], 0.0)
    nav_before_payment = b2_shares * open_price + b2_cash + b2_dividend_receivable
    b2_cash += dividend_payment
    b2_dividend_receivable -= dividend_payment
    nav_after_payment = b2_shares * open_price + b2_cash + b2_dividend_receivable
    if not math.isclose(nav_before_payment, nav_after_payment, abs_tol=1e-8):
        raise RuntimeError('El cobro de dividendos de B2 no conserva el NAV.')

    shares_before_ex_date = b2_shares
    dividend_net = shares_before_ex_date * row['dividend_per_share'] * (1 - DIVIDEND_WITHHOLDING)
    if dividend_net:
        payment_date = payment_available_by_ex_date.get(row['session_date'])
        if payment_date is None:
            raise RuntimeError('Falta una fecha de disponibilidad para un dividendo B2.')
        b2_dividend_receivable += dividend_net
        b2_pending_dividend_payments[payment_date] += dividend_net

    value_open_before_execution = b2_shares * open_price + b2_cash + b2_dividend_receivable
    effective_weight_open_before = (
        b2_shares * open_price / value_open_before_execution
        if value_open_before_execution > 0 else 0.0
    )
    order = b2_planned_order
    if order is not None:
        execution = execute_b1_buy(order, b2_cash, open_price)
        if execution['no_fill']:
            trade_executed = False
        else:
            b2_shares = execution['shares']
            b2_cash = execution['cash']
            trade_executed = True
            b2_investment_date = row['session_date']
    else:
        execution = {
            'fill_price': None, 'no_fill': False, 'trade_notional': 0.0,
            'commission': 0.0, 'iva': 0.0, 'total_cost': 0.0,
        }
        trade_executed = False

    portfolio_value_open = b2_shares * open_price + b2_cash + b2_dividend_receivable
    portfolio_value_close = b2_shares * close_price + b2_cash + b2_dividend_receivable
    effective_weight_open = b2_shares * open_price / portfolio_value_open
    effective_weight_close = b2_shares * close_price / portfolio_value_close
    b2_records.append({
        'session_date': row['session_date'],
        'signal_date': row['signal_date'],
        'order_decision_date': order['decision_date'] if order is not None else None,
        'target_weight': B2_INITIAL_TARGET_WEIGHT,
        'signal_close': row['signal_close'],
        'sigma_daily_for_limit': row['sigma_daily'],
        'limit_price': order['limit_price'] if order is not None else None,
        'cash_available_at_decision': order['cash_available_at_decision'] if order is not None else None,
        'cash_reserved': order['cash_reserved'] if order is not None else 0.0,
        'order_requested': order is not None,
        'order_side': 'buy' if order is not None else 'none',
        'order_quantity': order['order_quantity'] if order is not None else 0.0,
        'no_fill': execution['no_fill'],
        'trade_executed': trade_executed,
        'executed_shares': execution['shares'] if trade_executed else 0.0,
        'fill_price': execution['fill_price'],
        'trade_notional': execution['trade_notional'],
        'commission': execution['commission'],
        'iva': execution['iva'],
        'total_cost': execution['total_cost'],
        'open_price': open_price,
        'close_price': close_price,
        'shares_before_ex_date': shares_before_ex_date,
        'shares': b2_shares,
        'cash': b2_cash,
        'dividend_per_share': row['dividend_per_share'],
        'dividend_net': dividend_net,
        'dividend_payment': dividend_payment,
        'dividend_receivable': b2_dividend_receivable,
        'cash_interest': daily_interest,
        'nav_before_payment': nav_before_payment,
        'nav_after_payment': nav_after_payment,
        'external_cash_flow': 0.0,
        'portfolio_value_open': portfolio_value_open,
        'portfolio_value_close': portfolio_value_close,
        'effective_weight_open_before': effective_weight_open_before,
        'effective_weight_open': effective_weight_open,
        'effective_weight_close': effective_weight_close,
    })

    if trade_executed:
        b2_planned_order = None
    elif order is not None:
        if session_index + 1 >= len(b2_rows):
            raise RuntimeError('La orden inicial de B2 nunca se ejecutó dentro de la muestra.')
        next_row = b2_rows[session_index + 1]
        b2_planned_order = prepare_static_buy(
            b2_cash,
            next_row['signal_date'],
            next_row['signal_close'],
            next_row['sigma_daily'],
            B2_INITIAL_TARGET_WEIGHT,
        )

benchmark_b2 = pl.DataFrame(b2_records)

# Validaciones de calendario, causalidad, una sola compra y contabilidad.
if benchmark_b2['session_date'].to_list() != policy_p0['session_date'].to_list():
    raise ValueError('B2 no conserva el calendario común de P0/P1/P2.')
if benchmark_b2['session_date'].to_list() != benchmark_b1['session_date'].to_list():
    raise ValueError('B2 no conserva el calendario común de B1.')
if benchmark_b2['session_date'].to_list() != sorted(benchmark_b2['session_date'].to_list()):
    raise ValueError('Las fechas de B2 no están ordenadas.')
if benchmark_b2['session_date'].n_unique() != benchmark_b2.height:
    raise ValueError('B2 contiene fechas duplicadas.')
b2_orders = benchmark_b2.filter(pl.col('order_requested'))
if not (b2_orders['order_decision_date'] < b2_orders['session_date']).all():
    raise ValueError('B2 usó información futura para una orden.')
if benchmark_b2['trade_executed'].sum() != 1 or b2_investment_date is None:
    raise ValueError('B2 debe ejecutar exactamente una compra inicial.')
after_b2_investment = benchmark_b2.filter(pl.col('session_date') >= b2_investment_date)
if after_b2_investment['shares'].n_unique() != 1 or after_b2_investment['order_requested'].sum() != 1:
    raise ValueError('B2 generó operaciones posteriores a la compra inicial.')
if (
    not (benchmark_b2['shares'] >= 0).all()
    or not (benchmark_b2['cash'] >= -1e-8).all()
    or not (benchmark_b2['portfolio_value_close'] > 0).all()
    or not benchmark_b2['effective_weight_close'].is_between(0, 1).all()
):
    raise ValueError('B2 incumple límites long-only, efectivo o patrimonio.')
if not benchmark_b2.select(
    (pl.col('dividend_net') == pl.col('shares_before_ex_date') * pl.col('dividend_per_share') * (1 - DIVIDEND_WITHHOLDING)).all()
).item():
    raise ValueError('B2 contabilizó un dividendo de forma inconsistente.')
if (
    not math.isclose(benchmark_b2['dividend_net'].sum(), benchmark_b2['dividend_payment'].sum(), abs_tol=1e-8)
    or not math.isclose(benchmark_b2['dividend_receivable'][-1], 0.0, abs_tol=1e-8)
    or not math.isclose(benchmark_b2['external_cash_flow'].sum(), 0.0, abs_tol=1e-12)
):
    raise ValueError('B2 presenta doble conteo de dividendos o flujos externos.')
if not benchmark_b2.select(
    (pl.col('nav_before_payment') - pl.col('nav_after_payment')).abs().max() <= 1e-8
).item():
    raise ValueError('Un cobro de dividendos no conservó el NAV de B2.')
if set(benchmark_b1.columns).difference(benchmark_b2.columns):
    raise ValueError('B2 no conserva todas las variables esenciales de B1.')
if dataframe_sha256(benchmark_b1) != b1_hash_before_b2:
    raise RuntimeError('La construcción de B2 alteró el historial certificado de B1.')

investment_row = benchmark_b2.filter(pl.col('trade_executed')).row(0, named=True)
benchmark_b2_summary = pl.DataFrame([{
    'sessions': benchmark_b2.height,
    'investment_date': str(b2_investment_date),
    'trades_executed': int(benchmark_b2['trade_executed'].sum()),
    'initial_capital_usd': INITIAL_CAPITAL_USD,
    'final_portfolio_value': benchmark_b2['portfolio_value_close'][-1],
    'initial_effective_weight': investment_row['effective_weight_open'],
    'final_effective_weight': benchmark_b2['effective_weight_close'][-1],
}])
benchmark_b2_summary

### 5.4. Validación de comparabilidad de las cinco alternativas

Compartir una fuente de precios no basta para que dos backtests sean comparables. La comparabilidad temporal exige el mismo calendario; la contable exige valorar los mismos componentes del NAV bajo las mismas reglas; y la operativa exige documentar ejecución, costos, dividendos y efectivo.

Las diferencias de política son deliberadas: P0, P1 y P2 ajustan su exposición con volatility targeting; B1 conserva una inversión inicial completa en SPY; y B2 conserva una asignación inicial de 60% a SPY y 40% a efectivo. Estas exposiciones pueden divergir, pero las convenciones económicas que las contabilizan deben ser consistentes.

En esta auditoría, `total_cost` representa comisión e IVA registrados por la función de costos. El slippage también tiene efecto económico, pero entra mediante el precio de ejecución y ya afecta al NAV. Por tanto, no debe descontarse una segunda vez al atribuir costos en el futuro Bloque 07.

In [ ]:
if any(name not in globals() for name in [
    'policy_p0', 'policy_p1', 'policy_p2', 'benchmark_b1', 'benchmark_b2',
    'manifest', 'artifact_dir', 'benchmark_b1_summary', 'benchmark_b2_summary',
]):
    raise RuntimeError('Las secciones 5.1, 5.2 y 5.3 deben ejecutarse antes de la auditoría.')

EVALUATION_HISTORIES = {
    'P0 — Daily': policy_p0,
    'P1 — Banda 5 pp': policy_p1,
    'P2 — Banda 10 pp': policy_p2,
    'B1 — Buy & Hold SPY': benchmark_b1,
    'B2 — Asignación estática 60/40': benchmark_b2,
}
PANEL_COLUMNS = [
    'session_date', 'signal_date', 'target_weight', 'portfolio_value_open',
    'portfolio_value_close', 'effective_weight_close', 'cash', 'shares',
    'dividend_receivable', 'dividend_net', 'dividend_payment', 'cash_interest',
    'trade_executed', 'trade_notional', 'commission', 'iva', 'total_cost',
    'open_price', 'close_price', 'order_requested', 'order_side', 'no_fill',
]
ESSENTIAL_PANEL_COLUMNS = {
    'policy', 'session_date', 'portfolio_value_close', 'effective_weight_close',
    'cash', 'shares', 'dividend_receivable', 'trade_executed',
    'turnover_notional', 'commission', 'total_cost',
}
INITIAL_CAPITAL_USD_AUDIT = float(manifest['initial_capital']['net_usd'])

# Las copias hash permiten comprobar que el panel no altera B1 ni B2.
b1_hash_before_panel = dataframe_sha256(benchmark_b1)
b2_hash_before_panel = dataframe_sha256(benchmark_b2)

def validate_history(policy, history, reference_dates):
    missing = set(PANEL_COLUMNS).difference(history.columns)
    if missing:
        raise ValueError(f'{policy}: faltan columnas necesarias para la auditoría: {sorted(missing)}')
    dates = history['session_date'].to_list()
    if history.height != 2_206 or dates != reference_dates:
        raise ValueError(f'{policy}: el calendario no coincide con las 2,206 sesiones comunes.')
    if dates != sorted(dates) or len(dates) != len(set(dates)):
        raise ValueError(f'{policy}: hay fechas duplicadas o desordenadas.')
    if not math.isclose(
        float(history['cash_available_at_decision'][0]),
        INITIAL_CAPITAL_USD_AUDIT,
        rel_tol=0.0,
        abs_tol=1e-8,
    ):
        raise ValueError(f'{policy}: no parte del capital inicial USD documentado.')
    essential = history.select([
        pl.col(column).is_null().sum().alias(column)
        for column in ESSENTIAL_PANEL_COLUMNS.difference({'policy', 'turnover_notional'})
    ])
    if sum(essential.row(0)) != 0:
        raise ValueError(f'{policy}: tiene valores faltantes en variables esenciales.')
    if (
        not (history['portfolio_value_close'] > 0).all()
        or not (history['cash'] >= -1e-8).all()
        or not (history['shares'] >= 0).all()
        or not (history['dividend_receivable'] >= -1e-8).all()
    ):
        raise ValueError(f'{policy}: incumple límites de patrimonio, efectivo o posición.')
    nav_check = history.select(
        (
            pl.col('portfolio_value_close')
            - (pl.col('shares') * pl.col('close_price') + pl.col('cash') + pl.col('dividend_receivable'))
        ).abs().max().alias('maximum_nav_difference')
    ).item()
    if nav_check is None or nav_check > 1e-8:
        raise ValueError(f'{policy}: el NAV no incorpora consistentemente efectivo y dividendos por cobrar.')
    if (
        not math.isclose(history['dividend_net'].sum(), history['dividend_payment'].sum(), abs_tol=1e-8)
        or not math.isclose(history['dividend_receivable'][-1], 0.0, abs_tol=1e-8)
    ):
        raise ValueError(f'{policy}: los dividendos no quedaron conciliados.')
    if 'external_cash_flow' in history.columns and not math.isclose(
        history['external_cash_flow'].sum(), 0.0, abs_tol=1e-12
    ):
        raise ValueError(f'{policy}: se detectaron aportaciones o retiros externos.')

reference_dates = policy_p0['session_date'].to_list()
for policy, history in EVALUATION_HISTORIES.items():
    validate_history(policy, history, reference_dates)

# La procedencia de P0/P1/P2 debe seguir siendo la de los Parquet certificados.
for policy in ['P0 — Daily', 'P1 — Banda 5 pp', 'P2 — Banda 10 pp']:
    certificate = manifest['policies'][policy]
    path = artifact_dir / certificate['file']
    if sha256_file(path) != certificate['sha256']:
        raise ValueError(f'{policy}: el Parquet certificado no supera su verificación SHA-256.')
    if not math.isclose(
        EVALUATION_HISTORIES[policy]['portfolio_value_close'][-1],
        float(certificate['final_portfolio_value']),
        rel_tol=0.0,
        abs_tol=1e-8,
    ):
        raise ValueError(f'{policy}: el resultado terminal no coincide con su certificado.')

if not math.isclose(
    benchmark_b1['portfolio_value_close'][-1],
    benchmark_b1_summary['final_portfolio_value'][0],
    rel_tol=0.0,
    abs_tol=1e-8,
):
    raise ValueError('B1: el resultado terminal no coincide con su resumen validado.')
if not math.isclose(
    benchmark_b2['portfolio_value_close'][-1],
    benchmark_b2_summary['final_portfolio_value'][0],
    rel_tol=0.0,
    abs_tol=1e-8,
):
    raise ValueError('B2: el resultado terminal no coincide con su resumen validado.')

# Las políticas comparten precios de valoración sin ajustar y convenciones de ejecución.
reference_prices = policy_p0.select('session_date', 'open_price', 'close_price')
for policy, history in EVALUATION_HISTORIES.items():
    price_difference = history.join(reference_prices, on='session_date', how='inner', suffix='_reference').select(
        (pl.col('open_price') - pl.col('open_price_reference')).abs().max().alias('open_difference'),
        (pl.col('close_price') - pl.col('close_price_reference')).abs().max().alias('close_difference'),
    ).row(0, named=True)
    if any(value is None or value > 1e-10 for value in price_difference.values()):
        raise ValueError(f'{policy}: no utiliza la misma serie de precios operativos.')

def history_to_panel(policy, history):
    return history.select(
        pl.lit(policy).alias('policy'),
        *[pl.col(column) for column in PANEL_COLUMNS],
        pl.col('trade_notional').abs().alias('turnover_notional'),
    )

evaluation_panel = pl.concat([
    history_to_panel(policy, history)
    for policy, history in EVALUATION_HISTORIES.items()
]).sort('policy', 'session_date')

if evaluation_panel.height != 5 * len(reference_dates):
    raise ValueError('El panel común no contiene una fila por política y sesión.')
if ESSENTIAL_PANEL_COLUMNS.difference(evaluation_panel.columns):
    raise ValueError('El panel común no contiene todas las variables esenciales.')
if evaluation_panel.select([
    pl.col(column).is_null().sum().alias(column)
    for column in ESSENTIAL_PANEL_COLUMNS
]).row(0) != tuple(0 for _ in ESSENTIAL_PANEL_COLUMNS):
    raise ValueError('El panel común contiene valores faltantes en variables esenciales.')

if dataframe_sha256(benchmark_b1) != b1_hash_before_panel:
    raise RuntimeError('La auditoría alteró el historial de B1.')
if dataframe_sha256(benchmark_b2) != b2_hash_before_panel:
    raise RuntimeError('La auditoría alteró el historial de B2.')

comparability_audit = pl.DataFrame([
    {
        'policy': policy,
        'sessions': history.height,
        'first_date': str(history['session_date'][0]),
        'last_date': str(history['session_date'][-1]),
        'initial_capital_usd': INITIAL_CAPITAL_USD_AUDIT,
        'final_portfolio_value': history['portfolio_value_close'][-1],
        'validation_status': 'PASS',
    }
    for policy, history in EVALUATION_HISTORIES.items()
])
quality_gate_05 = 'PASS'
comparability_audit

Las cinco alternativas superan el quality gate de comparabilidad: comparten calendario, capital inicial disponible, moneda de valoración y convenciones contables y operativas documentadas. P0, P1 y P2 difieren deliberadamente en su regla de rebalanceo; B1 mantiene una exposición inicial completa a SPY; y B2 conserva su asignación inicial 60/40 sin rebalanceo.

Esto permite evaluar posteriormente sus resultados sobre una base común, pero no elimina dos límites importantes: las exposiciones y riesgos no son idénticos por diseño, y la evidencia continúa siendo retrospectiva. La comparación económica posterior deberá interpretar los costos registrados sin descontar de nuevo el slippage ya incorporado en el NAV.

## Bloque 06 — Evaluación de rendimiento y riesgo económico

### 6.1. Rendimientos y crecimiento geométrico

El NAV concentra el efecto económico de posiciones, efectivo, dividendos, intereses, comisiones y precios de ejecución. Por ello, los rendimientos se construyen a partir del NAV y no a partir de `adjusted_close` ni descontando costos por segunda vez.

Para cada política, con capital disponible inicial \(V_0\) y NAV de cierre \(V_t\), definimos:

$$R_t = \frac{V_t}{V_{t-1}} - 1, \qquad \ell_t = \log\left(\frac{V_t}{V_{t-1}}\right), \qquad W_t = \frac{V_t}{V_0}, \qquad R_{\mathrm{acum},t} = W_t - 1$$

Para la primera sesión se usa \(V_{t-1}=V_0\): el rendimiento recoge desde el capital disponible antes de la ejecución inicial hasta el NAV de cierre de esa primera fecha. En consecuencia, la base conceptual de riqueza es 1 antes de operar; el primer punto observado no se fuerza artificialmente a 1.

El rendimiento acumulado describe el cambio total en el periodo. El CAGR resume el crecimiento geométrico anual equivalente entre los extremos temporales:

$$\operatorname{CAGR} = \left(\frac{V_T}{V_0}\right)^{1/Y} - 1, \qquad Y = \frac{\text{días calendario entre la primera y la última sesión}}{365.2425}$$

Todas las alternativas parten del mismo capital disponible en USD, por lo que sus índices de riqueza comparten una base económica. Una mayor tasa de crecimiento, por sí sola, no demuestra que una política sea mejor: todavía faltan evaluar riesgo, pérdidas e incertidumbre estadística.

In [ ]:
import importlib.util
import subprocess
import sys

if 'evaluation_panel' not in globals() or 'EVALUATION_HISTORIES' not in globals():
    raise RuntimeError('La sección 5.4 debe ejecutarse antes de calcular rendimientos.')
if importlib.util.find_spec('plotly') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'plotly==6.2.0'])

import plotly.graph_objects as go

INITIAL_NAV_USD = float(manifest['initial_capital']['net_usd'])
PERFORMANCE_POLICIES = list(EVALUATION_HISTORIES)
first_session_date = evaluation_panel['session_date'].min()
last_session_date = evaluation_panel['session_date'].max()
elapsed_calendar_years = (last_session_date - first_session_date).days / 365.2425
if elapsed_calendar_years <= 0:
    raise ValueError('El intervalo temporal debe contener al menos un año calendario positivo.')

# Los hashes aseguran que el cálculo de rendimientos no modifica los historiales fuente.
history_hashes_before_performance = {
    policy: dataframe_sha256(history)
    for policy, history in EVALUATION_HISTORIES.items()
}

performance_parts = []
for policy in PERFORMANCE_POLICIES:
    policy_history = evaluation_panel.filter(pl.col('policy') == policy).sort('session_date')
    if policy_history.height != 2_206:
        raise ValueError(f'{policy}: número de sesiones inesperado en evaluation_panel.')
    performance_parts.append(
        policy_history.with_columns(
            (
                pl.col('portfolio_value_close')
                / pl.col('portfolio_value_close').shift(1).fill_null(INITIAL_NAV_USD)
                - 1
            ).alias('simple_return'),
            (pl.col('portfolio_value_close') / INITIAL_NAV_USD).alias('wealth_index'),
        ).with_columns(
            pl.col('simple_return').add(1).log().alias('log_return'),
            (pl.col('wealth_index') - 1).alias('cumulative_return'),
        )
    )

performance_panel = pl.concat(performance_parts).sort('policy', 'session_date')

# Conciliaciones matemáticas, con tolerancias explícitas para aritmética de punto flotante.
REL_TOL = 1e-10
ABS_TOL = 1e-10
performance_rows = []
for policy in PERFORMANCE_POLICIES:
    series = performance_panel.filter(pl.col('policy') == policy).sort('session_date')
    simple_returns = series['simple_return'].to_list()
    log_returns = series['log_return'].to_list()
    wealth_index = series['wealth_index'].to_list()
    final_nav = float(series['portfolio_value_close'][-1])
    final_ratio = final_nav / INITIAL_NAV_USD

    if len(simple_returns) != 2_206:
        raise ValueError(f'{policy}: número de rendimientos inconsistente.')
    if (
        not all(math.isfinite(value) and value > -1 for value in simple_returns)
        or not all(math.isfinite(value) for value in log_returns)
    ):
        raise ValueError(f'{policy}: existen rendimientos no finitos o inferiores a -100%.')
    compounded_return = math.prod(1 + value for value in simple_returns)
    if not math.isclose(compounded_return, final_ratio, rel_tol=REL_TOL, abs_tol=ABS_TOL):
        raise ValueError(f'{policy}: el producto de rendimientos no concilia con el NAV final.')
    if not math.isclose(sum(log_returns), math.log(final_ratio), rel_tol=REL_TOL, abs_tol=ABS_TOL):
        raise ValueError(f'{policy}: la suma de rendimientos logarítmicos no concilia con el NAV final.')
    if not math.isclose(wealth_index[0], 1 + simple_returns[0], rel_tol=REL_TOL, abs_tol=ABS_TOL):
        raise ValueError(f'{policy}: el índice no está normalizado al capital inicial antes de operar.')
    if not math.isclose(wealth_index[-1], final_ratio, rel_tol=REL_TOL, abs_tol=ABS_TOL):
        raise ValueError(f'{policy}: el índice de riqueza final no concilia con el NAV.')
    cagr = final_ratio ** (1 / elapsed_calendar_years) - 1
    if not math.isclose(
        (1 + cagr) ** elapsed_calendar_years,
        final_ratio,
        rel_tol=REL_TOL,
        abs_tol=ABS_TOL,
    ):
        raise ValueError(f'{policy}: el CAGR no concilia con los NAV extremos.')
    if 'external_cash_flow' in series.columns and not math.isclose(
        series['external_cash_flow'].sum(), 0.0, abs_tol=ABS_TOL
    ):
        raise ValueError(f'{policy}: se detectaron aportaciones o retiros externos.')
    performance_rows.append({
        'policy': policy,
        'initial_nav_usd': INITIAL_NAV_USD,
        'final_nav_usd': final_nav,
        'cumulative_return': final_ratio - 1,
        'CAGR': cagr,
    })

for policy, history in EVALUATION_HISTORIES.items():
    if dataframe_sha256(history) != history_hashes_before_performance[policy]:
        raise RuntimeError(f'El cálculo de rendimiento alteró el historial original de {policy}.')

performance_summary = pl.DataFrame(performance_rows)

PERFORMANCE_COLORS = {
    'P0 — Daily': '#1f77b4',
    'P1 — Banda 5 pp': '#ff7f0e',
    'P2 — Banda 10 pp': '#2ca02c',
    'B1 — Buy & Hold SPY': '#9467bd',
    'B2 — Asignación estática 60/40': '#8c564b',
}
wealth_figure = go.Figure()
for policy in PERFORMANCE_POLICIES:
    series = performance_panel.filter(pl.col('policy') == policy).sort('session_date')
    wealth_figure.add_trace(go.Scatter(
        x=series['session_date'].to_list(),
        y=series['wealth_index'].to_list(),
        mode='lines',
        name=policy,
        line=dict(color=PERFORMANCE_COLORS[policy], width=1.8),
    ))
wealth_figure.update_layout(
    title='Crecimiento de una unidad de capital inicial',
    xaxis_title='',
    yaxis_title='Índice de riqueza',
    height=450,
    legend_title='',
)
wealth_figure.show()
performance_summary

La tabla muestra diferencias en el crecimiento total y geométrico anual: B1 termina con el mayor índice de riqueza de la muestra, mientras que las demás trayectorias se mantienen en niveles distintos entre sí. La gráfica aporta además el momento y la persistencia de las separaciones o convergencias, información que no se observa sólo con el valor terminal.

Esta descripción no permite seleccionar una política superior. Aún faltan medidas de riesgo, pérdidas, costos atribuidos e incertidumbre estadística para interpretar los crecimientos observados.

### 6.2. Volatilidad realizada, drawdowns y recuperación del capital

La volatilidad realizada mide la dispersión observada de los rendimientos diarios de la cartera durante la muestra. No es lo mismo que la volatilidad pronosticada: esta última es una estimación disponible antes de una sesión, mientras que la primera resume lo que efectivamente ocurrió. Usamos la desviación estándar muestral y una convención de 252 sesiones anuales:

$$\sigma_{\mathrm{realizada}} = s(r_t)\sqrt{252}, \qquad s(r_t) = \sqrt{\frac{1}{n-1}\sum_{t=1}^{n}(r_t-\bar r)^2}$$

El drawdown compara el patrimonio con el máximo histórico que era conocido hasta esa misma fecha:

$$HWM_t = \max(HWM_{t-1}, V_t), \qquad DD_t = \frac{V_t}{HWM_t} - 1$$

Una cartera permanece *underwater* cuando su patrimonio sigue por debajo de su máximo histórico. Las pérdidas requieren recuperaciones porcentuales asimétricas: para un drawdown \(d<0\), el rendimiento necesario para volver al máximo es \(-d/(1+d)\). Por ello, la profundidad, la duración y el estado recuperado o abierto de cada episodio aportan información complementaria al crecimiento final.

In [ ]:
if 'performance_panel' not in globals() or 'performance_summary' not in globals():
    raise RuntimeError('La sección 6.1 debe ejecutarse antes de evaluar volatilidad y drawdowns.')

RISK_POLICIES = list(EVALUATION_HISTORIES)
RISK_REL_TOL = 1e-10
RISK_ABS_TOL = 1e-10

# Las auditorías de riesgo son de sólo lectura sobre los insumos validados.
history_hashes_before_risk = {
    policy: dataframe_sha256(history)
    for policy, history in EVALUATION_HISTORIES.items()
}
performance_hash_before_risk = dataframe_sha256(performance_panel)
reference_dates_risk = performance_panel.filter(
    pl.col('policy') == RISK_POLICIES[0]
).sort('session_date')['session_date'].to_list()

drawdown_records = []
risk_rows = []
episode_registry = {}

for policy in RISK_POLICIES:
    series = performance_panel.filter(pl.col('policy') == policy).sort('session_date')
    dates = series['session_date'].to_list()
    navs = series['portfolio_value_close'].to_list()
    returns = series['simple_return'].to_list()
    if len(dates) != 2_206 or dates != reference_dates_risk:
        raise ValueError(f'{policy}: el calendario no coincide con las sesiones comunes.')
    if not all(math.isfinite(value) for value in returns):
        raise ValueError(f'{policy}: los rendimientos no permiten calcular volatilidad realizada.')

    realized_vol = series['simple_return'].std(ddof=1) * math.sqrt(252)
    if not math.isfinite(realized_vol) or realized_vol < 0:
        raise ValueError(f'{policy}: volatilidad realizada inválida.')

    hwm_value = INITIAL_NAV_USD
    hwm_date = dates[0]
    active_episode = None
    episodes = []
    policy_drawdowns = []

    for date, nav in zip(dates, navs):
        if nav >= hwm_value - RISK_ABS_TOL:
            if active_episode is not None:
                active_episode['recovery_date'] = date
                active_episode['recovered'] = True
                active_episode['duration_sessions_to_recovery'] = active_episode['underwater_sessions']
                episodes.append(active_episode)
                active_episode = None
            if nav > hwm_value + RISK_ABS_TOL:
                hwm_value = nav
                hwm_date = date
            drawdown = 0.0
            underwater = False
        else:
            drawdown = nav / hwm_value - 1
            underwater = True
            if active_episode is None:
                active_episode = {
                    'peak_date': hwm_date,
                    'peak_value': hwm_value,
                    'trough_date': date,
                    'trough_value': nav,
                    'max_drawdown': drawdown,
                    'underwater_sessions': 0,
                    'recovery_date': None,
                    'recovered': False,
                    'duration_sessions_to_recovery': None,
                }
            active_episode['underwater_sessions'] += 1
            if drawdown < active_episode['max_drawdown']:
                active_episode['max_drawdown'] = drawdown
                active_episode['trough_date'] = date
                active_episode['trough_value'] = nav

        policy_drawdowns.append(drawdown)
        drawdown_records.append({
            'policy': policy,
            'session_date': date,
            'portfolio_value_close': nav,
            'high_water_mark': hwm_value,
            'high_water_mark_date': hwm_date,
            'drawdown': drawdown,
            'underwater': underwater,
        })

    if active_episode is not None:
        active_episode['duration_sessions_to_recovery'] = None
        episodes.append(active_episode)

    if any(value > RISK_ABS_TOL for value in policy_drawdowns):
        raise ValueError(f'{policy}: se detectó un drawdown positivo.')
    max_drawdown = min(policy_drawdowns)
    worst_episode = min(episodes, key=lambda episode: episode['max_drawdown']) if episodes else {
        'peak_date': dates[0],
        'trough_date': dates[0],
        'max_drawdown': 0.0,
        'recovery_date': dates[0],
        'recovered': True,
        'duration_sessions_to_recovery': 0,
        'underwater_sessions': 0,
    }
    if not math.isclose(max_drawdown, worst_episode['max_drawdown'], rel_tol=RISK_REL_TOL, abs_tol=RISK_ABS_TOL):
        raise ValueError(f'{policy}: el máximo drawdown no coincide con el mínimo de su serie.')
    if worst_episode['peak_date'] > worst_episode['trough_date']:
        raise ValueError(f'{policy}: el máximo previo ocurre después del mínimo del drawdown.')
    if worst_episode['recovery_date'] is not None:
        if worst_episode['recovery_date'] < worst_episode['trough_date']:
            raise ValueError(f'{policy}: la recuperación ocurre antes del mínimo del drawdown.')
        recovery_nav = series.filter(pl.col('session_date') == worst_episode['recovery_date'])['portfolio_value_close'][0]
        if recovery_nav < worst_episode['peak_value'] - RISK_ABS_TOL:
            raise ValueError(f'{policy}: la recuperación no alcanzó el máximo previo.')
    elif policy_drawdowns[-1] >= -RISK_ABS_TOL:
        raise ValueError(f'{policy}: un episodio recuperado fue marcado como abierto.')

    episode_registry[policy] = episodes
    risk_rows.append({
        'policy': policy,
        'realized_vol_annualized': realized_vol,
        'max_drawdown': max_drawdown,
        'max_dd_peak_date': worst_episode['peak_date'],
        'max_dd_trough_date': worst_episode['trough_date'],
        'recovery_date': worst_episode['recovery_date'],
        'max_dd_recovered': worst_episode['recovered'],
        'max_dd_duration_sessions': worst_episode['duration_sessions_to_recovery'],
        'longest_underwater_sessions': max(
            (episode['underwater_sessions'] for episode in episodes), default=0
        ),
    })

drawdown_panel = pl.DataFrame(drawdown_records).sort('policy', 'session_date')
risk_summary = pl.DataFrame(risk_rows)

if drawdown_panel.height != 5 * len(reference_dates_risk):
    raise ValueError('El panel de drawdowns no contiene una fila por política y sesión.')
for policy, history in EVALUATION_HISTORIES.items():
    if dataframe_sha256(history) != history_hashes_before_risk[policy]:
        raise RuntimeError(f'El análisis de riesgo alteró el historial original de {policy}.')
if dataframe_sha256(performance_panel) != performance_hash_before_risk:
    raise RuntimeError('El análisis de riesgo alteró performance_panel.')

drawdown_figure = go.Figure()
for policy in RISK_POLICIES:
    series = drawdown_panel.filter(pl.col('policy') == policy).sort('session_date')
    drawdown_figure.add_trace(go.Scatter(
        x=series['session_date'].to_list(),
        y=series['drawdown'].to_list(),
        mode='lines',
        name=policy,
        line=dict(color=PERFORMANCE_COLORS[policy], width=1.8),
    ))
drawdown_figure.add_hline(y=0, line=dict(color='#111111', width=1, dash='dash'))
drawdown_figure.update_layout(
    title='Drawdown histórico de las cinco políticas',
    xaxis_title='',
    yaxis=dict(title='Drawdown', tickformat='.0%'),
    height=450,
    legend_title='',
)
drawdown_figure.show()
risk_summary.select(
    'policy',
    'realized_vol_annualized',
    'max_drawdown',
    'max_dd_trough_date',
    'recovery_date',
    'longest_underwater_sessions',
)

La volatilidad realizada observada se ubicó entre 11.77% y 18.11% anualizados. Los máximos drawdowns abarcaron desde −16.05% hasta −32.57%; todos los episodios máximos se recuperaron dentro de la muestra. La gráfica muestra además que las diferencias de crecimiento observadas en 6.1 coexistieron con diferencias en profundidad y tiempo bajo máximo histórico.

Estas observaciones describen el historial realizado, pero todavía no establecen superioridad económica, significancia estadística ni una causa de las diferencias entre políticas.